# Exploración inicial de datos: brecha digital (EPH)
**Práctica Profesional I - Rita Josefina Juan Briones**

Este cuaderno te guía **paso a paso** para mirar el dataset y obtener la información que aparece en el trabajo: cuántas filas y columnas hay, cuántos datos faltan, qué valores tiene cada variable y un resumen de los números.

### Cómo se usa un cuaderno de Colab (primera vez)
- Un cuaderno está hecho de **celdas**. Las de texto (como esta) explican. Las de **código** tienen un botón de ▶ (play) a la izquierda.
- Para ejecutar una celda de código: hacé clic en ella y apretá el botón ▶ o las teclas **Shift + Enter**.
- **Ejecutá las celdas en orden**, de arriba hacia abajo, una por una. El resultado aparece justo debajo.
- Lo que está escrito después de un `#` es un **comentario**: es una explicación para vos, Python lo ignora.
- Si algo sale mal, no pasa nada: volvé a ejecutar la celda o empezá de nuevo desde arriba.

## Paso 0: Tener el archivo de datos en Colab
Necesitás el archivo `eph_tecnologia_microdatos.csv`. Hay **dos formas** de conseguirlo, elegí una:

**Opción A (subir el archivo):**
1. En el menú de la izquierda de Colab, hacé clic en el ícono de la **carpeta** 📁.
2. Hacé clic en el ícono de **subir archivo** (una hoja con una flecha hacia arriba) y elegí el CSV de tu computadora.
3. Esperá a que aparezca en la lista. *(Ojo: los archivos de Colab se borran cuando se cierra la sesión, así que si volvés otro día hay que subirlo de nuevo.)*

**Opción B (crearlo con código):** ejecutá la celda de abajo. Es el script de la cátedra y genera el mismo archivo.

In [ ]:
# OPCION B: crear el archivo CSV con el script de la catedra.
# Si ya subiste el CSV (opcion A), NO hace falta ejecutar esta celda.
import numpy as np
import pandas as pd

np.random.seed(42)          # semilla fija: asi siempre salen los mismos datos
n_registros = 1000          # cantidad de hogares

codusu_list = [f"HOG_EPH_2026_{i:04d}" for i in range(1, n_registros + 1)]
regiones = np.random.choice([1, 40, 41, 42, 43, 44], size=n_registros,
                            p=[0.35, 0.15, 0.15, 0.10, 0.15, 0.10])
integrantes = np.random.poisson(lam=2.3, size=n_registros) + 1
nivel_ed = np.random.choice([1, 2, 3, 4, 5, 9], size=n_registros,
                            p=[0.08, 0.22, 0.30, 0.25, 0.13, 0.02])

ingreso_base = {1: 280000, 2: 420000, 3: 580000, 4: 850000, 5: 1450000, 9: 500000}
itf_list = []
for ed in nivel_ed:
    mu = ingreso_base[ed]
    val = np.random.lognormal(mean=np.log(mu), sigma=0.45)
    itf_list.append(round(val, 2))
itf_array = np.array(itf_list)
decif_array = pd.qcut(itf_array, q=10, labels=list(range(1, 11)))

prob_pc, prob_inet = [], []
for ed, decil in zip(nivel_ed, decif_array):
    p_pc = 0.25 + (ed * 0.11) if ed != 9 else 0.45
    p_inet = 0.45 + (ed * 0.08) if ed != 9 else 0.65
    p_pc += (int(decil) - 5) * 0.03
    p_inet += (int(decil) - 5) * 0.025
    prob_pc.append(np.clip(p_pc, 0.10, 0.95))
    prob_inet.append(np.clip(p_inet, 0.25, 0.98))

iv12_3 = [np.random.choice([1, 2, 9], p=[p, 0.98 - p, 0.02]) for p in prob_pc]
ii7 = [np.random.choice([1, 2, 9], p=[p, 0.98 - p, 0.02]) for p in prob_inet]

data = {"CODUSU": codusu_list, "REGION": regiones, "IX_TOT": integrantes,
        "NIVEL_ED": nivel_ed, "ITF": itf_array, "DECIFR": decif_array,
        "IV12_3": iv12_3, "II7": ii7}
df_eph = pd.DataFrame(data)

# Se inducen valores faltantes a proposito (para practicar la limpieza)
mask_nulos_itf = np.random.choice([True, False], size=n_registros, p=[0.06, 0.94])
mask_nulos_pc = np.random.choice([True, False], size=n_registros, p=[0.03, 0.97])
df_eph.loc[mask_nulos_itf, "ITF"] = np.nan
df_eph.loc[mask_nulos_pc, "IV12_3"] = np.nan

df_eph.to_csv("eph_tecnologia_microdatos.csv", index=False)
print("Archivo creado:", df_eph.shape[0], "filas y", df_eph.shape[1], "columnas")

## Paso 1: Importar la librería para trabajar con datos
**¿Qué hacemos?** Python por sí solo no sabe manejar tablas. **pandas** es una librería (una "caja de herramientas") que sí sabe. La importamos con el apodo `pd`, que es la costumbre.

In [ ]:
# Importamos pandas con el apodo "pd"
import pandas as pd

## Paso 2: Cargar el archivo
**¿Qué hacemos?** Leemos el CSV y lo guardamos en una "variable" llamada `df` (por *DataFrame*, que es como pandas llama a una tabla). Desde ahora, `df` es tu tabla de datos.

In [ ]:
# Leemos el archivo CSV y lo guardamos en df
df = pd.read_csv("eph_tecnologia_microdatos.csv")

## Paso 3: ¿Cuántas filas y columnas tiene?
**¿Por qué?** Lo primero que hay que saber es el tamaño de lo que tenemos. Cada **fila** es un hogar y cada **columna** es una variable.

✅ **Deberías ver:** `(1000, 8)` → 1.000 hogares y 8 variables.

In [ ]:
# shape devuelve (cantidad de filas, cantidad de columnas)
print("Dimensiones:", df.shape)

## Paso 4: Mirar cómo se ve la tabla
**¿Por qué?** Antes de calcular nada, conviene *mirar* los datos con tus propios ojos. `head()` muestra las primeras 5 filas. `dtypes` muestra el **tipo de dato** de cada columna (números enteros, decimales, texto).

✅ **Fijate que:** `ITF` e `IV12_3` aparecen como `float64` (decimales). Eso pasa porque tienen valores faltantes (NaN).

In [ ]:
# Primeras 5 filas
df.head()

In [ ]:
# Tipo de dato de cada columna
df.dtypes

## Paso 5: Contar los datos faltantes (nulos)
**¿Por qué?** Los datos faltantes (**NaN**, "not a number") pueden arruinar los cálculos. Hay que saber dónde están y cuántos son antes de limpiar.

- `isna()` marca con `True` cada celda vacía.
- `sum()` cuenta cuántos `True` hay por columna.
- `mean() * 100` da el **porcentaje**.

✅ **Deberías ver:** `ITF` = 53 nulos (5,3 %) e `IV12_3` = 36 nulos (3,6 %). El resto, 0.

In [ ]:
# Cantidad de nulos por columna
print(df.isna().sum())

In [ ]:
# Porcentaje de nulos por columna (redondeado a 2 decimales)
print((df.isna().mean() * 100).round(2))

## Paso 6: Buscar los códigos "9 = sin respuesta"
**¿Por qué?** En la encuesta, el número **9 significa "sin respuesta"**. No aparece como dato vacío, pero tampoco es información real. Si lo dejamos, Python lo tomaría como un número más y distorsionaría todo.

Acá preguntamos: "¿cuántas veces vale 9 cada columna?"

✅ **Deberías ver:** `NIVEL_ED` = 17, `IV12_3` = 12, `II7` = 15.

In [ ]:
# (df[columnas] == 9) pone True donde vale 9; sum() los cuenta
print((df[["NIVEL_ED", "IV12_3", "II7"]] == 9).sum())

## Paso 7: Contar cuántos hogares hay en cada categoría
**¿Por qué?** Para variables de categorías (región, nivel educativo, tiene PC, etc.) lo que interesa es **cuántos casos hay en cada una**. `value_counts()` lo hace. Con `dropna=False` también cuenta los nulos, así no se nos escapan.

El `for` repite lo mismo para cada columna de la lista, así no hay que escribirlo cinco veces.

✅ **Algunos valores:** REGION 1 (GBA) = 365; DECIFR = 100 en cada decil.

In [ ]:
# Repetimos el conteo para cada variable de categorias
for col in ["REGION", "NIVEL_ED", "IV12_3", "II7", "DECIFR"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).sort_index())

## Paso 8: Resumen estadístico de las variables numéricas
**¿Por qué?** Para variables de números (cantidad de integrantes e ingreso) se calculan estadísticas que las resumen. `describe()` te da todo junto:
- **count**: cuántos datos válidos hay (no cuenta los nulos)
- **mean**: la media (el promedio)
- **std**: el desvío estándar (qué tan dispersos están los datos)
- **min / max**: el valor más chico y el más grande
- **25%, 50%, 75%**: los *percentiles*. El **50% es la mediana** (el valor del medio)

✅ **Lo importante en `ITF`:** la media (≈ 773.680) es bastante **mayor** que la mediana (≈ 604.132). Eso significa que pocos hogares con ingresos muy altos suben el promedio. Por eso para ingresos se prefiere la **mediana**.

In [ ]:
# Resumen estadistico de las dos variables numericas
df[["IX_TOT", "ITF"]].describe().round(2)

## Paso 9: Revisar si hay hogares repetidos
**¿Por qué?** El código `CODUSU` identifica cada hogar y no debería repetirse. Si hubiera repetidos, contaríamos dos veces el mismo hogar.

✅ **Deberías ver:** `0`.

In [ ]:
# duplicated() marca los repetidos; sum() los cuenta
print("CODUSU duplicados:", df["CODUSU"].duplicated().sum())

## Paso 10 (extra): Mirar la forma del ingreso con un gráfico
**¿Por qué?** Un gráfico dice más que muchos números. Un **histograma** agrupa los ingresos en "barras" y muestra cuántos hogares hay en cada rango. Vas a ver que la mayoría de los hogares está a la izquierda (ingresos más bajos) y que hay una cola larga hacia la derecha: eso es la **asimetría** que vimos entre media y mediana.

In [ ]:
import matplotlib.pyplot as plt   # libreria para hacer graficos

plt.figure(figsize=(8, 4))
plt.hist(df["ITF"].dropna(), bins=30, color="steelblue", edgecolor="white")  # dropna ignora los nulos
plt.title("Distribución del Ingreso Total Familiar (ITF)")
plt.xlabel("Ingreso (pesos)")
plt.ylabel("Cantidad de hogares")
plt.show()

## ✅ Resumen: lo que tenés que haber encontrado
| Qué miramos | Resultado esperado |
|---|---|
| Dimensiones | 1.000 filas y 8 columnas |
| Nulos en ITF | 53 (5,3 %) |
| Nulos en IV12_3 | 36 (3,6 %) |
| Código 9 en NIVEL_ED / IV12_3 / II7 | 17 / 12 / 15 |
| ITF: media y mediana | ≈ 773.680 y ≈ 604.132 |
| CODUSU duplicados | 0 |

Si te dan estos números, ¡lo hiciste bien! 🎉

**Próximo paso:** la *limpieza*: convertir los 9 en NaN, decidir qué hacer con los nulos y guardar el archivo limpio `df_clean.csv`.